# EDA — Response Clarity Classification (QEvasion / SemEval 2026 Task 6)

Loads the dataset from Hugging Face, inspects label distribution and text length,
and plots per-class statistics.

## 1. Imports & seed

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Seed:   {SEED}')

## 2. Config

In [ ]:
MODEL_NAME   = 'bert-base-uncased'
NUM_LABELS   = 3
MAX_LENGTH   = 256      # tokens (will experiment with 128 and 512 too)
BATCH_SIZE   = 16
EPOCHS       = 4
LR           = 2e-5
WARMUP_RATIO = 0.1
VAL_SIZE     = 0.15
PATIENCE     = 2        # early stopping patience (epochs)

LABEL2ID = {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print('Configuration:')
for k, v in dict(
    model=MODEL_NAME, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
    epochs=EPOCHS, lr=LR, val_size=VAL_SIZE, patience=PATIENCE
).items():
    print(f'  {k}: {v}')

## 3. Load dataset

In [ ]:
dataset = load_dataset('ailsntua/QEvasion')

raw_train = dataset['train'].to_pandas()
raw_test  = dataset['test'].to_pandas()

print(f'Train shape: {raw_train.shape}')
print(f'Test shape:  {raw_test.shape}')
print()
print('Label distribution (train):')
print(raw_train['clarity_label'].value_counts())

## 4. Label distribution & text length

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Label distribution
counts = raw_train['clarity_label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue', 'salmon', 'seagreen'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 20, str(val), ha='center')

# Question length (words)
q_lens = raw_train['question'].str.split().map(len)
axes[1].hist(q_lens, bins=40, color='steelblue', edgecolor='white')
axes[1].set_title(f'Question Length (words)\nMean={q_lens.mean():.1f}, Max={q_lens.max()}')
axes[1].set_xlabel('Words')

# Answer length (words)
a_lens = raw_train['interview_answer'].str.split().map(len)
axes[2].hist(a_lens, bins=40, color='salmon', edgecolor='white')
axes[2].set_title(f'Answer Length (words)\nMean={a_lens.mean():.1f}, Max={a_lens.max()}')
axes[2].set_xlabel('Words')

plt.tight_layout()
plt.savefig('eda_bert.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClass imbalance ratio (Ambivalent / Clear Non-Reply): {counts["Ambivalent"] / counts["Clear Non-Reply"]:.1f}x')